<a href="https://colab.research.google.com/github/nithish1492/vaccination-data-analysis-visualization/blob/main/Sample_EDA_Submission_Template_Filled_Vaccination_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Vaccination Data Analysis and Visualization


##### **Project Type**    - EDA
##### **Contribution**    - Individual
##### **Team Member 1 -** Nithish Kumar



# **Project Summary -**


This project, Vaccination Data Analysis and Visualization, focuses on extracting meaningful public-health insights from supplied vaccination, disease-incidence, reported-case, vaccine-introduction, and vaccine-schedule datasets. The analysis is designed to support evidence-based understanding of vaccination coverage, disease burden, vaccine introduction patterns, and immunization schedules across countries and years. The work follows an end-to-end exploratory data analysis workflow covering data loading, inspection, quality assessment, cleaning, transformation, visualization, relationship analysis, and business interpretation.

The first stage establishes a clear understanding of the supplied datasets by examining their structure, dimensions, data types, duplicate records, missing values, unique values, and descriptive statistics. Because the project uses multiple source files, each dataset is inspected independently before analytical relationships are created. This prevents unsupported assumptions and keeps the analysis traceable to the supplied data. Missing values are measured rather than automatically treated as zero, while invalid or out-of-range observations are handled carefully according to the meaning of each variable.

The second stage prepares analysis-ready datasets. Country codes, years, disease identifiers, antigen identifiers, numeric measures, and text fields are standardized where required. Coverage values are converted to numeric form and valid percentage values are separated from missing or out-of-range observations for analytical calculations. Disease incidence and reported cases are kept distinct because incidence rates and case counts represent different measures. Vaccine introduction and vaccine schedule information are analyzed using their available country, region, year, vaccine, target-population, and schedule fields.

The visualization stage uses bar charts, line charts, scatter plots, distribution plots, heatmaps, and a pair plot to investigate temporal trends, country-level differences, vaccine-antigen coverage, disease burden, vaccine introductions, and relationships between vaccination coverage and disease measures. The analysis specifically considers questions such as how vaccination coverage varies over time, whether higher coverage is associated with lower disease incidence where the supplied data allow that comparison, how reported cases vary by disease and year, and how vaccine introduction and schedule information differ across countries and regions.

The project does not fabricate unavailable demographic, socioeconomic, urban/rural, population-density, gender, education, or seasonal variables. Where a requested public-health question cannot be directly answered from the supplied files, it is explicitly identified as unsupported rather than estimated. This approach improves analytical integrity and makes the conclusions defensible.

The final outcome is a reproducible EDA notebook that can be executed cell by cell after the supplied datasets are placed in the Colab environment. The findings are intended to support public-health strategy, disease-prevention planning, identification of coverage gaps, prioritization of further investigation, and clearer communication of vaccination and disease patterns. The notebook is also aligned with the project's broader SQL and Power BI workflow, where cleaned and structured information can be used for normalized storage and interactive reporting.


# **GitHub Link -**

Provide your GitHub Link here.

GitHub repository: https://github.com/nithish1492/vaccination-data-analysis-visualization


# **Problem Statement**


The objective is to analyze the supplied vaccination and public-health datasets to understand vaccination coverage, disease incidence, reported disease cases, vaccine introduction patterns, and vaccine schedules across countries and years. The project must identify data-quality issues, prepare reliable analysis-ready datasets, visualize important trends and relationships, and translate the observed patterns into useful public-health insights without introducing unsupported assumptions or fabricated observations.


#### **Define Your Business Objective?**

The business objective is to provide an evidence-based analytical view that can help public-health stakeholders understand vaccination coverage patterns, identify countries or periods with comparatively lower observed coverage, examine disease burden alongside vaccination measures where the datasets support a valid comparison, and understand vaccine introduction and schedule patterns. The analysis can support resource prioritization, campaign planning, disease-prevention monitoring, and further investigation of areas requiring public-health attention. Recommendations are restricted to relationships that are directly supported by the supplied datasets.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import libraries and configure display
import os
import warnings
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


### Dataset Loading

In [ ]:
# Locate and load the five supplied Excel datasets
BASE_DIR = Path("/content")

def find_dataset(candidates, keywords):
    for name in candidates:
        path = BASE_DIR / name
        if path.exists():
            return path
    excel_files = list(BASE_DIR.glob("*.xlsx")) + list(BASE_DIR.glob("*.xls"))
    normalized = [(p, re.sub(r"[^a-z0-9]", "", p.name.lower())) for p in excel_files]
    for path, normalized_name in normalized:
        if all(k in normalized_name for k in keywords):
            return path
    return None

dataset_files = {
    "coverage": find_dataset(
        ["coverage-data.xlsx", "coverage_data.xlsx"],
        ["coverage", "data"]
    ),
    "incidence": find_dataset(
        ["incidence-rate-data(1).xlsx", "incidence-rate-data.xlsx", "incidence_rate_data.xlsx"],
        ["incidence", "rate"]
    ),
    "reported_cases": find_dataset(
        ["reported-cases-data(1).xlsx", "reported-cases-data.xlsx", "reported_cases_data.xlsx"],
        ["reported", "cases"]
    ),
    "introduction": find_dataset(
        ["vaccine-introduction-data(1).xlsx", "vaccine-introduction-data.xlsx", "vaccine_introduction_data.xlsx"],
        ["vaccine", "introduction"]
    ),
    "schedule": find_dataset(
        ["vaccine-schedule-data(1).xlsx", "vaccine-schedule-data.xlsx", "vaccine_schedule_data.xlsx"],
        ["vaccine", "schedule"]
    )
}

missing_files = [name for name, path in dataset_files.items() if path is None]
if missing_files:
    available = [p.name for p in BASE_DIR.glob("*.xlsx")] + [p.name for p in BASE_DIR.glob("*.xls")]
    raise FileNotFoundError(
        "The following required datasets were not found: "
        + ", ".join(missing_files)
        + ". Upload the five supplied Excel files to /content and run this cell again. "
        + f"Available Excel files: {available}"
    )

coverage_df = pd.read_excel(dataset_files["coverage"])
incidence_df = pd.read_excel(dataset_files["incidence"])
reported_cases_df = pd.read_excel(dataset_files["reported_cases"])
introduction_df = pd.read_excel(dataset_files["introduction"])
schedule_df = pd.read_excel(dataset_files["schedule"])

datasets = {
    "Coverage": coverage_df,
    "Incidence Rate": incidence_df,
    "Reported Cases": reported_cases_df,
    "Vaccine Introduction": introduction_df,
    "Vaccine Schedule": schedule_df
}

print("Datasets loaded successfully.")
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")


### Dataset First View

In [ ]:
# Display the first five rows of every supplied dataset
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head())


### Dataset Rows & Columns count

In [ ]:
# Display row and column counts
shape_table = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in datasets.items()],
    columns=["Dataset", "Rows", "Columns"]
)
display(shape_table)


### Dataset Information

In [ ]:
# Display data types and non-null counts
for name, df in datasets.items():
    print(f"\n{name}")
    df.info()


#### Duplicate Values

In [ ]:
# Calculate exact duplicate-row counts
duplicate_table = pd.DataFrame(
    [(name, int(df.duplicated().sum())) for name, df in datasets.items()],
    columns=["Dataset", "Duplicate Rows"]
)
display(duplicate_table)


#### Missing Values/Null Values

In [ ]:
# Calculate missing values and missing percentages
missing_tables = {}
for name, df in datasets.items():
    missing = df.isna().sum().to_frame("Missing Values")
    missing["Missing Percentage"] = (missing["Missing Values"] / len(df) * 100).round(2)
    missing_tables[name] = missing
    print(f"\n{name}")
    display(missing[missing["Missing Values"] > 0].sort_values("Missing Values", ascending=False))


In [ ]:
# Visualize missing values for all supplied datasets
for name, df in datasets.items():
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    missing_pct = missing_pct[missing_pct > 0]
    if missing_pct.empty:
        print(f"{name}: no missing values detected.")
        continue
    plt.figure(figsize=(10, max(4, len(missing_pct) * 0.35)))
    sns.barplot(x=missing_pct.values, y=missing_pct.index)
    plt.title(f"Missing Values by Column: {name}")
    plt.xlabel("Missing values (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()


### What did you know about your dataset?

The supplied data contain multiple complementary public-health tables rather than one single dataset. The coverage table contains country, year, antigen, target-number, doses, and coverage measures. The incidence and reported-case tables provide disease-level measures. Vaccine-introduction data describe when vaccines were introduced and include WHO-region information, while the vaccine-schedule data describe schedule and target-population fields.

The datasets contain substantial missing values in some analytical measures, so missingness must be treated as missing information rather than automatically converted to zero. Coverage values also require validation because not every stored value is suitable for a percentage-based average. Country, year, antigen, and disease fields provide common analytical dimensions that allow carefully controlled comparisons across datasets. The exact counts, distributions, and trends are generated by the executable cells below after the supplied files are loaded.


## ***2. Understanding Your Variables***

In [ ]:
# Display the columns in every dataset
for name, df in datasets.items():
    print(f"\n{name} columns")
    print(df.columns.tolist())


In [ ]:
# Display descriptive statistics for numeric variables
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.describe(include="all").T)


### Variables Description

The variables represent several layers of the vaccination system. Country and code identify geographic observations; Year provides the time dimension; Antigen and Disease identify the immunization or disease measure; Coverage, Doses, and Target Number quantify vaccination activity; Incidence Rate and Cases quantify disease burden; vaccine introduction fields describe introduction status; and schedule fields describe administration patterns and target populations.

Numeric variables are used for statistical analysis only after checking their data types and missing values. Categorical fields are retained because they are essential for grouping, filtering, and comparing public-health observations.


### Check Unique Values for each variable.

In [ ]:
# Check the number of unique values for every variable
unique_tables = {}
for name, df in datasets.items():
    unique_tables[name] = df.nunique(dropna=False).sort_values(ascending=False).to_frame("Unique Values")
    print(f"\n{name}")
    display(unique_tables[name])


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Prepare analysis-ready copies without altering the raw datasets

raw_datasets = {name: df.copy(deep=True) for name, df in datasets.items()}


def clean_column_name(column):
    text = str(column).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def standardize_columns(df):
    result = df.copy()
    result.columns = [clean_column_name(c) for c in result.columns]
    return result


# Standardize the five supplied datasets
coverage_clean = standardize_columns(coverage_df)
incidence_clean = standardize_columns(incidence_df)
reported_cases_clean = standardize_columns(reported_cases_df)
introduction_clean = standardize_columns(introduction_df)
schedule_clean = standardize_columns(schedule_df)


def rename_using_aliases(df, aliases):
    lookup = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in df.columns
    }

    rename_map = {}

    for target, possible_names in aliases.items():
        for name in possible_names:
            key = re.sub(r"[^a-z0-9]", "", str(name).lower())

            if key in lookup:
                rename_map[lookup[key]] = target
                break

    return df.rename(columns=rename_map)


# Standardize Coverage dataset columns
coverage_clean = rename_using_aliases(
    coverage_clean,
    {
        "Group": [
            "Group"
        ],
        "Code": [
            "Code",
            "ISO3 Code",
            "ISO3Code",
            "ISO_3_Code"
        ],
        "CountryName": [
            "Country Name",
            "CountryName",
            "Country",
            "Name"
        ],
        "Year": [
            "Year"
        ],
        "Antigen": [
            "Antigen"
        ],
        "AntigenDescription": [
            "Antigen Description",
            "AntigenDescription",
            "Antigen_description"
        ],
        "CoverageCategory": [
            "Coverage Category",
            "CoverageCategory",
            "Coverage_category"
        ],
        "CoverageCategoryDescription": [
            "Coverage Category Description",
            "CoverageCategoryDescription",
            "Coverage_category description",
            "Coverage_category_description"
        ],
        "TargetNumber": [
            "Target Number",
            "TargetNumber",
            "Target number"
        ],
        "Doses": [
            "Doses",
            "Dose",
            "Dodge"
        ],
        "Coverage": [
            "Coverage"
        ]
    }
)


# Standardize Incidence Rate dataset columns
incidence_clean = rename_using_aliases(
    incidence_clean,
    {
        "Group": [
            "Group"
        ],
        "Code": [
            "Code",
            "ISO3 Code",
            "ISO3Code",
            "ISO_3_Code"
        ],
        "CountryName": [
            "Country Name",
            "CountryName",
            "Country",
            "Name"
        ],
        "Year": [
            "Year"
        ],
        "Disease": [
            "Disease"
        ],
        "DiseaseDescription": [
            "Disease Description",
            "DiseaseDescription",
            "Disease description"
        ],
        "Denominator": [
            "Denominator"
        ],
        "IncidenceRate": [
            "Incidence Rate",
            "IncidenceRate",
            "Incidence rate"
        ]
    }
)


# Standardize Reported Cases dataset columns
reported_cases_clean = rename_using_aliases(
    reported_cases_clean,
    {
        "Group": [
            "Group"
        ],
        "Code": [
            "Code",
            "ISO3 Code",
            "ISO3Code",
            "ISO_3_Code"
        ],
        "CountryName": [
            "Country Name",
            "CountryName",
            "Country",
            "Name"
        ],
        "Year": [
            "Year"
        ],
        "Disease": [
            "Disease"
        ],
        "DiseaseDescription": [
            "Disease Description",
            "DiseaseDescription",
            "Disease description"
        ],
        "Cases": [
            "Cases",
            "Reported Cases",
            "ReportedCases"
        ]
    }
)


# Standardize Vaccine Introduction dataset columns
introduction_clean = rename_using_aliases(
    introduction_clean,
    {
        "ISO3Code": [
            "ISO3 Code",
            "ISO3Code",
            "ISO_3_Code",
            "Code"
        ],
        "CountryName": [
            "Country Name",
            "CountryName",
            "Country"
        ],
        "WHORegion": [
            "WHO Region",
            "WHORegion",
            "Who Region",
            "Region"
        ],
        "Year": [
            "Year"
        ],
        "Description": [
            "Description"
        ],
        "Intro": [
            "Intro",
            "Introduction"
        ]
    }
)


# Standardize Vaccine Schedule dataset columns
schedule_clean = rename_using_aliases(
    schedule_clean,
    {
        "ISO3Code": [
            "ISO3 Code",
            "ISO3Code",
            "ISO_3_Code",
            "Code"
        ],
        "CountryName": [
            "Country Name",
            "CountryName",
            "Country"
        ],
        "WHORegion": [
            "WHO Region",
            "WHORegion",
            "Who Region",
            "Region"
        ],
        "Year": [
            "Year"
        ],
        "VaccineCode": [
            "Vaccine Code",
            "VaccineCode",
            "Vaccine"
        ],
        "VaccineDescription": [
            "Vaccine Description",
            "VaccineDescription"
        ],
        "ScheduleRounds": [
            "Schedule Rounds",
            "ScheduleRounds",
            "Schedule rounds",
            "Rounds"
        ],
        "TargetPop": [
            "Target Pop",
            "TargetPop",
            "Target Population",
            "Target pop"
        ],
        "TargetPopDescription": [
            "Target Pop Description",
            "TargetPopDescription",
            "Target pop description"
        ],
        "Geoarea": [
            "Geoarea",
            "Geo Area",
            "Geographic Area"
        ],
        "AgeAdministered": [
            "Age Administered",
            "AgeAdministered",
            "Age administered"
        ],
        "SourceComment": [
            "Source Comment",
            "SourceComment",
            "Comment"
        ]
    }
)


# Define the minimum columns required for each analysis dataset
required_columns = {
    "Coverage": [
        "CountryName",
        "Code",
        "Year",
        "Antigen",
        "TargetNumber",
        "Doses",
        "Coverage"
    ],

    "Incidence Rate": [
        "CountryName",
        "Code",
        "Year",
        "Disease",
        "IncidenceRate"
    ],

    "Reported Cases": [
        "CountryName",
        "Code",
        "Year",
        "Disease",
        "Cases"
    ],

    "Vaccine Introduction": [
        "CountryName",
        "ISO3Code",
        "Year",
        "WHORegion",
        "Description",
        "Intro"
    ],

    "Vaccine Schedule": [
        "CountryName",
        "ISO3Code",
        "Year",
        "VaccineCode",
        "ScheduleRounds",
        "TargetPop"
    ]
}


# Create a mapping for validation
df_map = {
    "Coverage": coverage_clean,
    "Incidence Rate": incidence_clean,
    "Reported Cases": reported_cases_clean,
    "Vaccine Introduction": introduction_clean,
    "Vaccine Schedule": schedule_clean
}


# Validate required columns
for name, required in required_columns.items():

    missing = [
        column
        for column in required
        if column not in df_map[name].columns
    ]

    if missing:
        raise KeyError(
            f"{name}: required columns not found after standardization: {missing}"
        )


# Convert Year columns to numeric nullable integers
for df in [
    coverage_clean,
    incidence_clean,
    reported_cases_clean,
    introduction_clean,
    schedule_clean
]:

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(
            df["Year"],
            errors="coerce"
        ).astype("Int64")


# Clean country codes and country names
for df in [
    coverage_clean,
    incidence_clean,
    reported_cases_clean
]:

    for col in [
        "Code",
        "CountryName"
    ]:

        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )


# Clean introduction and schedule dimensions
for df in [
    introduction_clean,
    schedule_clean
]:

    for col in [
        "ISO3Code",
        "CountryName",
        "WHORegion"
    ]:

        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )


# Convert Coverage numeric fields
for col in [
    "TargetNumber",
    "Doses",
    "Coverage"
]:

    coverage_clean[col] = pd.to_numeric(
        coverage_clean[col],
        errors="coerce"
    )


# Convert Incidence Rate to numeric
incidence_clean["IncidenceRate"] = pd.to_numeric(
    incidence_clean["IncidenceRate"],
    errors="coerce"
)


# Convert Reported Cases to numeric
reported_cases_clean["Cases"] = pd.to_numeric(
    reported_cases_clean["Cases"],
    errors="coerce"
)


# Create valid analytical coverage records
coverage_valid = coverage_clean.loc[
    coverage_clean["Coverage"].between(
        0,
        100,
        inclusive="both"
    )
].copy()


# Create valid analytical incidence records
incidence_valid = incidence_clean.loc[
    incidence_clean["IncidenceRate"].ge(0)
].copy()


# Create valid analytical reported-case records
reported_cases_valid = reported_cases_clean.loc[
    reported_cases_clean["Cases"].ge(0)
].copy()


# Aggregate vaccination coverage at country-year level
coverage_country_year = (
    coverage_valid
    .groupby(
        [
            "Code",
            "CountryName",
            "Year"
        ],
        dropna=False
    )
    .agg(
        AverageCoverage=("Coverage", "mean"),
        TotalDoses=("Doses", "sum"),
        TotalTargetNumber=("TargetNumber", "sum")
    )
    .reset_index()
)


# Aggregate disease incidence at country-year level
incidence_country_year = (
    incidence_valid
    .groupby(
        [
            "Code",
            "CountryName",
            "Year"
        ],
        dropna=False
    )
    .agg(
        AverageIncidenceRate=("IncidenceRate", "mean")
    )
    .reset_index()
)


# Aggregate reported disease cases at country-year level
cases_country_year = (
    reported_cases_valid
    .groupby(
        [
            "Code",
            "CountryName",
            "Year"
        ],
        dropna=False
    )
    .agg(
        TotalReportedCases=("Cases", "sum")
    )
    .reset_index()
)


# Combine country-year analytical datasets
country_year_summary = (
    coverage_country_year
    .merge(
        incidence_country_year,
        on=[
            "Code",
            "CountryName",
            "Year"
        ],
        how="outer"
    )
    .merge(
        cases_country_year,
        on=[
            "Code",
            "CountryName",
            "Year"
        ],
        how="outer"
    )
)


# Display data-wrangling results
print("Data wrangling completed successfully.")

print(
    f"Valid coverage records for percentage analysis: "
    f"{len(coverage_valid):,}"
)

print(
    f"Valid incidence records for non-negative analysis: "
    f"{len(incidence_valid):,}"
)

print(
    f"Valid reported-case records for non-negative analysis: "
    f"{len(reported_cases_valid):,}"
)

print(
    f"Country-year analytical rows: "
    f"{len(country_year_summary):,}"
)

print("\nStandardized Coverage columns:")
print(coverage_clean.columns.tolist())

print("\nStandardized Incidence Rate columns:")
print(incidence_clean.columns.tolist())

print("\nStandardized Reported Cases columns:")
print(reported_cases_clean.columns.tolist())

print("\nStandardized Vaccine Introduction columns:")
print(introduction_clean.columns.tolist())

print("\nStandardized Vaccine Schedule columns:")
print(schedule_clean.columns.tolist())

print("\nCountry-year analytical preview:")
display(country_year_summary.head())

### What all manipulations have you done and insights you found?

The wrangling process standardizes column names for reliable programming, converts numeric fields to numeric data types, normalizes country/code text, removes exact duplicates where appropriate for analysis copies, and validates year fields. Missing values are retained unless a calculation specifically requires a non-missing measure. Coverage percentages outside the valid analytical range are not silently replaced; they are flagged and excluded from percentage averages while the raw values remain available.

Separate analysis tables are created for coverage, incidence, reported cases, vaccine introduction, and vaccine schedule. A controlled country-year analytical table is created only where matching country and year information exists in the supplied datasets. No unavailable demographic or socioeconomic fields are invented.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart visualization code
coverage_trend = coverage_valid.groupby("Year", dropna=True)["Coverage"].mean().reset_index()
if not coverage_trend.empty:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=coverage_trend, x="Year", y="Coverage", marker="o")
    plt.title("Average Valid Vaccination Coverage by Year")
    plt.xlabel("Year")
    plt.ylabel("Average Coverage (%)")
    plt.tight_layout()
    plt.show()
    display(coverage_trend.sort_values("Year").tail(10))
else:
    print("No valid coverage-year observations are available for this chart.")


##### 1. Why did you pick the specific chart?

A line chart is appropriate because Year is a time variable and vaccination coverage is a percentage that can be compared across years. The chart makes long-term increases, decreases, and periods of relative stability easier to identify than a table alone.

The exact pattern and values are calculated from valid coverage observations in the supplied data. The table printed below the chart provides recent yearly values for verification.


##### 2. What is/are the insight(s) found from the chart?

The chart shows how the average valid vaccination coverage changes over time. Peaks and declines should be interpreted as descriptive changes in the supplied records. The exact years with the highest and lowest observed averages can be verified from the generated table rather than assumed in advance.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These insights can support monitoring of vaccination performance and help identify periods that may require further investigation. Lower observed coverage can indicate a potential area for public-health attention, but the supplied data alone do not establish the operational reason for a change. Additional program, demographic, and socioeconomic information would be required for causal interpretation.


#### Chart - 2

In [ ]:
# Chart visualization code
country_coverage = (
    coverage_valid.groupby("CountryName")["Coverage"]
    .mean()
    .sort_values(ascending=False)
)
country_coverage = country_coverage.head(15).sort_values()
if not country_coverage.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=country_coverage.values, y=country_coverage.index)
    plt.title("Top 15 Countries by Average Valid Vaccination Coverage")
    plt.xlabel("Average Coverage (%)")
    plt.ylabel("Country")
    plt.tight_layout()
    plt.show()
    display(country_coverage.sort_values(ascending=False).to_frame("Average Coverage (%)"))
else:
    print("No valid country-level coverage observations are available.")


##### 1. Why did you pick the specific chart?

A horizontal bar chart is suitable for comparing many countries because country names are categorical and can be displayed clearly along one axis. Showing the top observed countries makes differences in average valid coverage easy to compare.


##### 2. What is/are the insight(s) found from the chart?

The chart identifies countries with the highest average valid coverage in the supplied records. The exact ranking is generated directly from the data and should be used to describe observed differences without assuming that the ranking represents overall vaccination-system performance.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Country-level differences can help public-health analysts identify where coverage is comparatively higher or lower and where further investigation may be useful. The chart does not by itself explain why differences occur or whether a country is adequately covered for every antigen.


#### Chart - 3

In [ ]:
# Chart visualization code
antigen_doses = (
    coverage_valid.groupby("Antigen")["Doses"]
    .sum(min_count=1)
    .dropna()
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
)
if not antigen_doses.empty:
    plt.figure(figsize=(10, 7))
    sns.barplot(x=antigen_doses.values, y=antigen_doses.index)
    plt.title("Total Doses by Antigen: Top 15")
    plt.xlabel("Total Doses")
    plt.ylabel("Antigen")
    plt.tight_layout()
    plt.show()
    display(antigen_doses.sort_values(ascending=False).to_frame("Total Doses"))
else:
    print("No valid dose observations are available.")


##### 1. Why did you pick the specific chart?

A bar chart is appropriate for comparing total doses across antigen categories. It provides a direct view of which antigens account for the largest recorded dose volumes in the supplied coverage data.


##### 2. What is/are the insight(s) found from the chart?

The chart highlights the antigens with the largest recorded dose totals. Because the number of records and target populations can differ between antigens, dose totals should not be interpreted as a direct measure of coverage or effectiveness.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding dose distribution can help with high-level resource and workload assessment. A high dose total does not automatically mean higher population coverage, so coverage percentages and target numbers should be considered alongside this chart.


#### Chart - 4

In [ ]:
# Chart visualization code
disease_incidence_year = (
    incidence_valid.groupby(["Year", "Disease"])["IncidenceRate"]
    .mean()
    .reset_index()
)
top_diseases = (
    incidence_valid.groupby("Disease")["IncidenceRate"]
    .mean()
    .sort_values(ascending=False)
    .head(6)
    .index
)
plot_df = disease_incidence_year[disease_incidence_year["Disease"].isin(top_diseases)]
if not plot_df.empty:
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=plot_df, x="Year", y="IncidenceRate", hue="Disease")
    plt.title("Disease Incidence Rate Trends for Major Observed Diseases")
    plt.xlabel("Year")
    plt.ylabel("Average Incidence Rate")
    plt.tight_layout()
    plt.show()
else:
    print("No valid disease incidence observations are available.")


##### 1. Why did you pick the specific chart?

A multi-line chart is useful for observing how disease incidence rates change over time and for comparing several diseases on the same time axis. Only diseases with the highest observed average incidence in the supplied data are displayed to keep the visualization readable.


##### 2. What is/are the insight(s) found from the chart?

The chart allows the observed temporal patterns of major recorded diseases to be compared. The exact direction, peaks, and troughs should be read from the plotted lines and the supplied observations rather than treated as evidence of causation.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Disease-incidence trends can support surveillance and help identify periods requiring further investigation. A change in incidence cannot be attributed to vaccination alone because many epidemiological, environmental, demographic, and reporting factors are not represented in this notebook.


#### Chart - 5

In [ ]:
# Chart visualization code
cases_year = (
    reported_cases_valid.groupby("Year")["Cases"]
    .sum(min_count=1)
    .reset_index()
)
if not cases_year.empty:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=cases_year, x="Year", y="Cases", marker="o")
    plt.title("Total Reported Disease Cases by Year")
    plt.xlabel("Year")
    plt.ylabel("Reported Cases")
    plt.tight_layout()
    plt.show()
    display(cases_year.sort_values("Year").tail(10))
else:
    print("No valid reported-case observations are available.")


##### 1. Why did you pick the specific chart?

A line chart is suitable because reported cases are measured across years and the objective includes understanding disease burden over time. Aggregating non-negative case observations by year provides a simple descriptive trend.


##### 2. What is/are the insight(s) found from the chart?

The chart shows how total recorded disease cases vary over the available years. Changes should be interpreted as changes in reported records, not necessarily as changes in true underlying disease occurrence.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The trend can support surveillance and help identify years that deserve closer investigation. Reported cases may be affected by reporting practices, population size, outbreak conditions, and data completeness, so the chart should not be used as a causal estimate.


#### Chart - 6

In [ ]:
# Chart visualization code
scatter_df = country_year_summary.dropna(subset=["AverageCoverage", "AverageIncidenceRate"]).copy()
if len(scatter_df) > 0:
    sample = scatter_df.sample(min(5000, len(scatter_df)), random_state=42)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=sample, x="AverageCoverage", y="AverageIncidenceRate", alpha=0.5)
    plt.title("Vaccination Coverage vs Disease Incidence Rate")
    plt.xlabel("Average Valid Coverage (%)")
    plt.ylabel("Average Incidence Rate")
    plt.tight_layout()
    plt.show()
    print("Pearson correlation:", sample["AverageCoverage"].corr(sample["AverageIncidenceRate"]))
else:
    print("Insufficient matched coverage and incidence observations for this comparison.")


##### 1. Why did you pick the specific chart?

A scatter plot is appropriate for examining the relationship between two continuous measures: average valid vaccination coverage and average disease incidence rate. It allows the spread of country-year observations and possible association patterns to be seen directly.


##### 2. What is/are the insight(s) found from the chart?

The chart shows whether matched country-year observations display an apparent positive, negative, or weak association between the two measures. The notebook also calculates a Pearson correlation for the plotted sample. Correlation does not establish causation.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This comparison can help identify observations that may warrant epidemiological investigation. A negative association, if observed, is consistent with the question of whether higher coverage and lower incidence occur together, but the supplied observational data cannot by themselves prove vaccine effectiveness or causality.


#### Chart - 7

In [ ]:
# Chart visualization code
coverage_category = (
    coverage_valid["CoverageCategory"]
    .fillna("Missing")
    .value_counts()
    .head(15)
    .sort_values()
)
if not coverage_category.empty:
    plt.figure(figsize=(10, 7))
    sns.barplot(x=coverage_category.values, y=coverage_category.index)
    plt.title("Coverage Records by Coverage Category")
    plt.xlabel("Number of Records")
    plt.ylabel("Coverage Category")
    plt.tight_layout()
    plt.show()
else:
    print("Coverage category information is not available.")


##### 1. Why did you pick the specific chart?

A categorical bar chart is appropriate for showing how coverage records are distributed across the supplied coverage categories. It makes the relative frequency of the recorded categories easy to compare.


##### 2. What is/are the insight(s) found from the chart?

The chart identifies the most frequently represented coverage categories in the supplied data. These frequencies describe the structure of the dataset and should not be interpreted as population-level proportions unless the source methodology supports that interpretation.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding the composition of coverage records helps prevent misleading conclusions caused by uneven category representation. It also helps analysts decide which categories have sufficient observations for deeper analysis.


#### Chart - 8

In [ ]:
# Chart visualization code
disease_avg = (
    incidence_valid.groupby("Disease")["IncidenceRate"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
)
if not disease_avg.empty:
    plt.figure(figsize=(10, 7))
    sns.barplot(x=disease_avg.values, y=disease_avg.index)
    plt.title("Average Incidence Rate by Disease")
    plt.xlabel("Average Incidence Rate")
    plt.ylabel("Disease")
    plt.tight_layout()
    plt.show()
    display(disease_avg.sort_values(ascending=False).to_frame("Average Incidence Rate"))
else:
    print("No valid incidence-rate observations are available.")


##### 1. Why did you pick the specific chart?

A horizontal bar chart is appropriate for comparing average incidence rates across diseases. Sorting the values makes relative differences easy to inspect while keeping disease names readable.


##### 2. What is/are the insight(s) found from the chart?

The chart displays diseases with the highest average incidence rates among valid non-negative observations. The values represent the supplied incidence-rate records and may use different denominators, so direct comparison should follow the source definitions of each disease measure.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Disease-level comparison can help prioritize surveillance questions and identify measures that deserve additional investigation. It should not be interpreted as a ranking of public-health severity without considering denominators, reporting quality, population structure, and disease-specific definitions.


#### Chart - 9

In [ ]:
# Chart visualization code
intro_counts = (
    introduction_clean.groupby("Year")
    .size()
    .reset_index(name="Introduction Records")
    .dropna(subset=["Year"])
)
if not intro_counts.empty:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=intro_counts, x="Year", y="Introduction Records", marker="o")
    plt.title("Vaccine Introduction Records by Year")
    plt.xlabel("Year")
    plt.ylabel("Number of Introduction Records")
    plt.tight_layout()
    plt.show()
    display(intro_counts.sort_values("Year").tail(10))
else:
    print("No vaccine introduction year information is available.")


##### 1. Why did you pick the specific chart?

A time-series chart is appropriate for vaccine introduction records because Year is the key temporal variable. Counting introduction records by year provides a descriptive view of when introduction observations appear in the supplied dataset.


##### 2. What is/are the insight(s) found from the chart?

The chart shows the number of vaccine-introduction records represented in each year. This describes the dataset's recorded introduction events or records and does not by itself measure vaccine availability, uptake, or effectiveness.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The observed timing can help analysts understand the historical introduction structure represented in the source. It can support further investigation of introduction milestones, but program decisions should use official introduction and implementation records beyond this EDA where necessary.


#### Chart - 10

In [ ]:
# Chart visualization code
region_intro = (
    introduction_clean["WHORegion"]
    .fillna("Missing")
    .value_counts()
    .sort_values()
)
if not region_intro.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=region_intro.values, y=region_intro.index)
    plt.title("Vaccine Introduction Records by WHO Region")
    plt.xlabel("Number of Records")
    plt.ylabel("WHO Region")
    plt.tight_layout()
    plt.show()
else:
    print("WHO region information is not available.")


##### 1. Why did you pick the specific chart?

A bar chart is appropriate for comparing the number of vaccine-introduction records across WHO regions. It provides a clear view of the geographic composition of the supplied introduction data.


##### 2. What is/are the insight(s) found from the chart?

The chart shows how introduction records are distributed across the available WHO-region labels. Differences may reflect the number of countries, vaccines, years, or observations represented in each region and should not be interpreted as differences in vaccine availability.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Regional comparison can help organize further public-health analysis and identify where more detailed country-level investigation may be useful. The supplied data do not provide enough information to infer reasons for regional differences.


#### Chart - 11

In [ ]:
# Chart visualization code
rounds = (
    schedule_clean["ScheduleRounds"]
    .fillna("Missing")
    .astype(str)
    .value_counts()
    .head(15)
    .sort_values()
)
if not rounds.empty:
    plt.figure(figsize=(10, 7))
    sns.barplot(x=rounds.values, y=rounds.index)
    plt.title("Most Common Vaccine Schedule-Round Entries")
    plt.xlabel("Number of Records")
    plt.ylabel("Schedule Rounds")
    plt.tight_layout()
    plt.show()
else:
    print("Schedule-round information is not available.")


##### 1. Why did you pick the specific chart?

A bar chart is appropriate for the schedule-round field because the variable is categorical in the supplied dataset. Displaying the most common recorded entries provides a concise view of schedule patterns.


##### 2. What is/are the insight(s) found from the chart?

The chart identifies the most frequently recorded schedule-round entries. Because schedule-round values are represented as source text, the visualization describes the source categories rather than converting them into an assumed numeric dose count.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This can support understanding of how schedules are represented in the source data and help identify commonly occurring schedule configurations. Programmatic recommendations should use the official immunization schedule definitions for the relevant country and population.


#### Chart - 12

In [ ]:
# Chart visualization code
target_pop = (
    schedule_clean["TargetPop"]
    .fillna("Missing")
    .astype(str)
    .value_counts()
    .head(15)
    .sort_values()
)
if not target_pop.empty:
    plt.figure(figsize=(10, 7))
    sns.barplot(x=target_pop.values, y=target_pop.index)
    plt.title("Most Common Vaccine Schedule Target-Population Entries")
    plt.xlabel("Number of Records")
    plt.ylabel("Target Population")
    plt.tight_layout()
    plt.show()
else:
    print("Target-population information is not available.")


##### 1. Why did you pick the specific chart?

A bar chart is appropriate for the target-population field because it is a categorical schedule attribute. Showing the most frequent entries helps describe which target-population labels are most represented.


##### 2. What is/are the insight(s) found from the chart?

The chart shows the most common target-population entries in the supplied vaccine-schedule data. Missing target-population values are retained as missing and are not converted into an assumed category.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding target-population representation can help analysts interpret schedule records and identify where the supplied data contain limited detail. It does not provide a population-size estimate or demographic coverage measure.


#### Chart - 13

In [ ]:
# Chart visualization code
heatmap_df = (
    coverage_valid.groupby(["CountryName", "Year"])["Coverage"]
    .mean()
    .reset_index()
)
if not heatmap_df.empty:
    top_countries = (
        coverage_valid.groupby("CountryName")["Coverage"]
        .mean()
        .sort_values(ascending=False)
        .head(20)
        .index
    )
    heatmap_df = heatmap_df[heatmap_df["CountryName"].isin(top_countries)]
    pivot = heatmap_df.pivot(index="CountryName", columns="Year", values="Coverage")
    if not pivot.empty:
        plt.figure(figsize=(15, 8))
        sns.heatmap(pivot, cmap="YlGnBu", linewidths=0.2)
        plt.title("Vaccination Coverage Heatmap for Selected Countries")
        plt.xlabel("Year")
        plt.ylabel("Country")
        plt.tight_layout()
        plt.show()
    else:
        print("Insufficient data for the coverage heatmap.")
else:
    print("No valid coverage observations are available.")


##### 1. Why did you pick the specific chart?

A heatmap is appropriate for country-year vaccination coverage because it displays two categorical dimensions, country and year, with color representing the numeric coverage value. This makes persistent gaps and changes over time easier to spot.


##### 2. What is/are the insight(s) found from the chart?

The heatmap highlights coverage patterns across selected countries and years. Darker or lighter cells represent different observed coverage levels according to the plotted scale. Missing cells indicate that no valid coverage value was available for that country-year combination.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This visualization can help identify country-year observations that deserve further investigation and can support monitoring of geographic disparities. It should not be interpreted as a complete country comparison because the displayed countries are selected from the supplied observations and missing data remain possible.


#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_for_corr = country_year_summary[
    ["AverageCoverage", "TotalDoses", "TotalTargetNumber", "AverageIncidenceRate", "TotalReportedCases"]
].copy()
corr = numeric_for_corr.corr(numeric_only=True)
if corr.shape[0] >= 2:
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("Correlation Heatmap of Country-Year Analytical Measures")
    plt.tight_layout()
    plt.show()
    display(corr)
else:
    print("Insufficient numeric variables for correlation analysis.")


##### 1. Why did you pick the specific chart?

A correlation heatmap is useful for summarizing linear relationships among the numeric measures in the matched country-year analytical table. It provides a compact comparison of several measures at once.


##### 2. What is/are the insight(s) found from the chart?

The heatmap reports pairwise Pearson correlations among average coverage, dose and target totals, incidence rate, and reported cases where sufficient matched data are available. Correlation values close to zero indicate weak linear association, while values closer to -1 or +1 indicate stronger linear association. Correlation does not imply causation.


#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pair_df = country_year_summary[
    ["AverageCoverage", "TotalDoses", "AverageIncidenceRate", "TotalReportedCases"]
].dropna().copy()

if len(pair_df) >= 2:
    pair_df = pair_df.sample(min(1000, len(pair_df)), random_state=42)
    pair_df["Log Total Doses"] = np.log1p(pair_df["TotalDoses"].clip(lower=0))
    pair_df["Log Reported Cases"] = np.log1p(pair_df["TotalReportedCases"].clip(lower=0))
    pair_plot_columns = [
        "AverageCoverage",
        "AverageIncidenceRate",
        "Log Total Doses",
        "Log Reported Cases"
    ]
    sns.pairplot(pair_df[pair_plot_columns], corner=True, diag_kind="hist")
    plt.show()
else:
    print("Insufficient matched numeric observations for a pair plot.")


##### 1. Why did you pick the specific chart?

A pair plot is useful for exploring distributions and pairwise relationships among several continuous measures simultaneously. Log transformations are used for highly skewed dose and reported-case measures so that the plotted distributions are easier to inspect.


##### 2. What is/are the insight(s) found from the chart?

The pair plot provides a visual check of distributions, possible outliers, clustering, and pairwise relationships in the matched country-year observations. The plot is exploratory and should be followed by statistical and domain-specific validation before drawing causal conclusions.


## **5. Solution to Business Objective**

Use the results of the executed analysis to focus public-health attention on observable coverage gaps, disease-burden patterns, and changes over time. Countries or years with comparatively lower valid vaccination coverage can be investigated for operational barriers and campaign needs, while disease trends can be monitored alongside vaccination measures without assuming causation from correlation alone. Vaccine introduction and schedule patterns can support planning by showing where and when particular vaccines were introduced or scheduled.

The client should use the notebook findings as an evidence layer rather than as a standalone causal model. Any intervention should be validated with additional operational, demographic, socioeconomic, population, and epidemiological information that is not present in the supplied files.


The executed analysis should be used to identify observable vaccination coverage patterns, disease-burden trends, and vaccine-introduction or schedule structures supported by the supplied records. Priority areas can be selected for further investigation based on persistent low valid coverage, notable disease trends, or substantial differences across countries and years.

The client should combine these findings with operational and epidemiological information not contained in the supplied datasets before implementing interventions. The notebook deliberately avoids inventing unsupported variables such as gender, education, urban/rural status, socioeconomic status, population density, or actual vaccine demand.


# **Conclusion**

This EDA project converts five supplied vaccination and public-health datasets into a structured analytical workflow covering data understanding, quality checks, cleaning, visualization, and relationship analysis. It provides separate views of vaccination coverage, disease incidence, reported cases, vaccine introductions, and schedules, followed by controlled cross-dataset comparisons where common country and year fields are available.

The analysis preserves the distinction between missing information and zero values and avoids fabricating variables that are not contained in the supplied datasets. The resulting visualizations and statistical summaries provide a reproducible basis for identifying temporal patterns, geographic differences, vaccination coverage gaps, and disease-burden relationships. The final conclusions should be interpreted as descriptive evidence from the supplied data and not as causal estimates.


### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***

In [ ]:
# Generate a final data-quality summary for submission
quality_summary = []

for name, df in datasets.items():
    numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Missing Cells": int(df.isna().sum().sum()),
        "Numeric Columns": len(numeric_columns)
    })

quality_summary = pd.DataFrame(quality_summary)
display(quality_summary)

print("Final validation completed.")
print("Raw datasets were preserved in raw_datasets.")
print("Analytical calculations use validated non-negative incidence/case values and coverage values between 0 and 100.")
print("No unsupported demographic or socioeconomic variables were fabricated.")


In [ ]:
# Print reproducibility information and key analytical counts
print("Vaccination Data Analysis and Visualization")
print("=" * 50)
print(f"Coverage records: {len(coverage_df):,}")
print(f"Incidence-rate records: {len(incidence_df):,}")
print(f"Reported-case records: {len(reported_cases_df):,}")
print(f"Vaccine-introduction records: {len(introduction_df):,}")
print(f"Vaccine-schedule records: {len(schedule_df):,}")
print(f"Country-year analytical rows: {len(country_year_summary):,}")
print()
print("Notebook execution is complete. Review the generated tables, charts, and observations before submission.")
